<a href="https://colab.research.google.com/github/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/blob/main/code/NRE5615_Wk2_Demo_Iris_Decision_Tree_From_CSV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iris Dataset: Decision Tree Classification / Demo / Template

**Objective:** Learn the basic supervised learning workflow using a simple CSV file and a Decision Tree model.

By the end of this activity you will be able to:

1. Read the data from `iris.csv`.
2. Check the dataset.
3. Split the data into **training**, **validation**, and **test** sets.
4. Train several Decision Tree models.
5. Use the **validation set** to choose the best model.
6. Use the **test set only at the end** for a final check.
7. Also check prediction for random input

This same workflow can be followed in **Orange**: load data → choose target → train Tree model → evaluate → inspect confusion matrix.


## Before you start in Google Colab

This notebook uses the file **`iris.csv`**.

The first row of the CSV file should contain column names:

```text
sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,iris_species
```

### Here is brief infograph of the dataset
![Iris dataset info](https://raw.githubusercontent.com/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/main/data/iris_data_info.png)


## 1. Read/Import the data

You have two options to import/get the file into the notebook

1. Upload `iris.csv` from your desktop
2. Get it from a URL directly to the notebook

Click the run button ▶️ on the left side for eaither option 1 or 2 to run the cell.

Once you complete the step you should be able to see the first few rows of the dataset.

In [ ]:
# Option (1)
from google.colab import files
import pandas as pd
uploaded = files.upload()
df = pd.read_csv("iris.csv")
df.head()

In [ ]:
# Option (2) No need to execute this if option (1) worked
import pandas as pd
url = "https://raw.githubusercontent.com/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/main/data/iris.csv"
df = pd.read_csv(url)
df.head()

## Why this is supervised learning

The Iris dataset includes flower measurements and a known flower species for each example.

- **Input features:** sepal length, sepal width, petal length, petal width
- **Known answer / target label:** iris species

Because the correct answer is already included during training, this is a **supervised learning** problem.

## Libraries used in this notebook

We use a few common Python libraries:

- **pandas**: reads and organizes the CSV data as a table.
- **matplotlib**: creates simple plots.
- **train_test_split**: splits the data into training, validation, and test sets.
- **DecisionTreeClassifier**: creates the Decision Tree model.
- **plot_tree**: draws the trained Decision Tree.
- **accuracy_score, confusion_matrix, classification_report**: help us evaluate the model.



In [ ]:
# Import libraries

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

## 2. Quick data check

Before training a model, always check the data.

We will check:

- Number of rows and columns
- Column names
- Missing values
- Number of examples in each species

Can also visualize a few attributes?

Possible prompt:
"Code to create a distribution bar chart for the variable/field sepel length using only mathplotlib ?"

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(list(df.columns))

print("\nMissing values by column:")
print(df.isna().sum())

print("\nNumber of examples per species:")
print(df["iris_species"].value_counts())

display(df.head())

## 3. Separate input features and target label

The model needs two parts:

- **X** = input features used to make a prediction
- **y** = target label, or the known answer

Here, the target label is `iris_species`.

In [ ]:
feature_columns = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm"
]

target_column = "iris_species"

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Input features:")
display(X.head())

print("Target labels:")
display(y.head())

## 4. Split into training, validation, and test sets

We will use three sets:

| Split | Purpose |
|---|---|
| **Training set** | Used to teach the model |
| **Validation set** | Used to compare models and choose model settings |
| **Test set** | Used only at the end for a final performance check |

For this activity:

- 60% training
- 20% validation
- 20% test

We use `stratify=y` so each split has a similar mix of iris species.

In [ ]:
# First split: 60% training and 40% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

# Second split: temporary data into 20% validation and 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape[0], "rows")
print("Validation set:", X_val.shape[0], "rows")
print("Test set:", X_test.shape[0], "rows")

## 6. Train a simple Decision Tree model

A Decision Tree uses a series of simple rules to classify examples.

One important setting is `max_depth`:

- A small depth creates a simpler tree.
- A large depth creates a more complex tree.
- A very complex tree may memorize the training data instead of learning general patterns.

We can try several tree depths and compare them using the validation set accuracy.

Change the depth to 2,3,4,5,6 and run multiple time, and try to select the best model

In [ ]:
# Try several tree depths by changing the dept =
depth = 1

results = []
model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
        )
# Train on the training set
model.fit(X_train, y_train)

# Predict on training and validation sets
train_pred = model.predict(X_train)
val_pred = model.predict(X_val)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, train_pred)
val_accuracy = accuracy_score(y_val, val_pred)

# Show results
print("Decision Tree Results")
print("---------------------")
print(f"Max depth used: {depth}")
print(f"Actual tree depth: {model.get_depth()}")
print(f"Number of leaves: {model.get_n_leaves()}")
print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Validation accuracy: {val_accuracy:.3f}")

## Model evaluation terms: TP, FP, precision, recall, and F1

A **confusion matrix** shows how the model's predictions compare with the correct answers.

For one class at a time, such as **setosa**, we can think about:

- **True Positive (TP):** The flower is setosa, and the model predicts setosa.
- **False Positive (FP):** The flower is not setosa, but the model predicts setosa.
- **False Negative (FN):** The flower is setosa, but the model predicts a different species.
- **True Negative (TN):** The flower is not setosa, and the model also predicts not setosa.

For a dataset with more than two classes, these ideas are applied **one class at a time**.

Common evaluation measures:

- **Accuracy:** Overall, how often was the model correct?
- **Precision:** When the model predicted a class, how often was it right?
- **Recall:** Of the actual examples in a class, how many did the model find?
- **F1-score:** A balance between precision and recall.

Formulas:

```text
Precision = TP / (TP + FP)
Recall    = TP / (TP + FN)
F1-score  = 2 × (Precision × Recall) / (Precision + Recall)
```

## 7. Evaluate the selected model on the validation set

Now we train the selected model and examine:

- Validation accuracy
- Validation confusion matrix
- Precision, recall, and F1-score

In [ ]:
# Train the selected model on the training set
selected_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

selected_model.fit(X_train, y_train)

# Predict validation labels
val_pred = selected_model.predict(X_val)

# Validation accuracy
val_accuracy = accuracy_score(y_val, val_pred)
print("Validation accuracy:", round(val_accuracy, 3))

# Use the class order learned by the model
class_names = list(selected_model.classes_)

# Confusion matrix
cm_val = confusion_matrix(y_val, val_pred, labels=class_names)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=class_names
)

disp.plot()
plt.title("Validation Confusion Matrix")
plt.xticks(rotation=30)
plt.show()

print("\nValidation classification report:")
print(classification_report(y_val, val_pred, zero_division=0))

## 8. Visualize the selected Decision Tree

This plot shows the rules learned by the model.

For example, a tree may use petal length or petal width to separate different iris species.

In [ ]:
plt.figure(figsize=(16, 8))

plot_tree(
    selected_model,
    feature_names=feature_columns,
    class_names=class_names,
    filled=True,
    rounded=True
)

plt.title("Selected Decision Tree")
plt.show()

## 9. Final evaluation on the test set

Now that we have selected the model using the validation set, we evaluate it one final time on the **test set**.

Important rule:

> Do not use the test set to choose the model. Use it only at the end.

This gives a more honest estimate of how the model may perform on new data.

In [ ]:
# Test set predictions
test_pred = selected_model.predict(X_test)

# Test accuracy
test_accuracy = accuracy_score(y_test, test_pred)
print("Final test accuracy:", round(test_accuracy, 3))

# Test confusion matrix
cm_test = confusion_matrix(y_test, test_pred, labels=class_names)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_test,
    display_labels=class_names
)

disp.plot()
plt.title("Final Test Confusion Matrix")
plt.xticks(rotation=30)
plt.show()

print("\nFinal test classification report:")
print(classification_report(y_test, test_pred, zero_division=0))

## 10. Make a prediction for a new flower

Now we can try the model on one new example.

Change the numbers below and run the cell again to see how the prediction changes.

In [ ]:
new_flower = pd.DataFrame([{
    "sepal_length_cm": 2.1,
    "sepal_width_cm": 3.9,
    "petal_length_cm": 1.4,
    "petal_width_cm": 0.6
}])

# Keep only the columns used by the model, in the correct order
new_flower = new_flower[feature_columns]

prediction = selected_model.predict(new_flower)

print("Predicted iris species:", prediction[0])

### Reflection questions

1. What are the input features in this dataset?
2. What is the target label?
3. Why is this a supervised learning problem?
4. Which `max_depth` was selected using the validation set?
5. Did the validation and test results look similar?
6. Why should we not use the test set to choose the model?
7. What does the confusion matrix show?

### Connection to Orange

How to do this in Orange (Optional), the same workflow would look like this:

1. Load `iris.csv` using the **CSV File Import** widget or **URL**.
2. Check the data using the **Data Table** widget.
3. Make sure `iris_species` is treated as the target/class variable.
4. Train a **Tree** model.
5. Use **Test & Score** to compare results.
6. Use **Confusion Matrix** to inspect mistakes.
7. Use final test results only after deciding on the model.


### When you are finished, **disconnect and delete** the Colab runtime so that it does not use compute time when you are not using it.

This option is avaialble when you click arrow lik button in top righhand side of the page where we have RAM / Disk indicated.